# 08_quantization: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/08_quantization/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Encode each subvector as the address of a learned codeword.

Concepts: Encode each subvector as the address of a learned codeword.
Run: uv run python modules/08_quantization/examples/01_from_scratch.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

import numpy as np

from common.index_lessons import unit_catalog
from tinyvec.pq import PQ


def main():
    _, matrix, q = unit_catalog(320)
    pq = PQ(matrix, parts=8, centroids=16)
    codes = pq.codes[0]
    restored = np.concatenate([book[code] for book, code in zip(pq.books, codes, strict=True)])
    print(
        "first code",
        codes,
        "reconstruction error",
        np.linalg.norm(matrix[0] - restored),
        "payload bytes",
        pq.payload_bytes,
    )
    print("nearest by asymmetric lookup", pq.candidates(q, 5))
    assert pq.codes.shape == (320, 8)


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/08_quantization/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Train a residual IVF-PQ index with FAISS.

    Concepts: Train a residual IVF-PQ index with FAISS.
    Run: uv run --extra libs python modules/08_quantization/examples/02_production.py
    Expected: Inspect the printed IDs or measurements and the checked invariant.
    CPU: seconds on the included small workload. Uses local data; no Docker or model download.
    """

    # course-extra: libs
    import faiss
    import numpy as np

    from common.index_lessons import unit_catalog

    def main():
        products, matrix, q = unit_catalog(640)
        faiss.omp_set_num_threads(1)
        index = faiss.IndexIVFPQ(
            faiss.IndexFlatIP(matrix.shape[1]), matrix.shape[1], 8, 8, 3, faiss.METRIC_INNER_PRODUCT
        )
        index.train(matrix)
        index.add(matrix)
        index.nprobe = 4
        _, labels = index.search(q[None, :], 40)
        candidates = labels[0][labels[0] >= 0]
        reranked = candidates[np.argsort(-(matrix[candidates] @ q), kind="stable")[:10]]
        print(
            "compressed leaders",
            products.iloc[candidates[:10]].id.tolist(),
            "after rescoring",
            products.iloc[reranked].id.tolist(),
        )
        print("packed residual bytes per row", index.code_size)
        assert len(reranked) == 10

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/08_quantization/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure code bytes, oversampling and trained prefix rankings.

Concepts: Measure code bytes, oversampling and trained prefix rankings.
Run: uv run python modules/08_quantization/examples/03_measure.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

from common.index_lessons import prefix_results, trial
from common.teaching import evidence


def main():
    observations = {
        "compression": [
            trial(kind, oversample=s) for kind in ("scalar", "binary", "pq") for s in (1, 8)
        ],
        "qwen_prefixes": prefix_results(),
    }
    print(observations)
    evidence("lesson-08_quantization", {"observations": observations}, tier=2)


if __name__ == "__main__":
    main()

## 04_native_codes

Source: `modules/08_quantization/examples/04_native_codes.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Compare FAISS scalar and product quantizers.

    Run: uv run --extra libs python modules/08_quantization/examples/04_native_codes.py
    Expected: compare the reported ranking and verify the final assertion.
    CPU: seconds, included data, no Docker or model downloads.
    """

    # course-extra: libs
    import faiss

    from common.index_lessons import unit_catalog

    def main():
        _, matrix, q = unit_catalog(640)
        faiss.omp_set_num_threads(1)
        for index in [
            faiss.IndexScalarQuantizer(
                matrix.shape[1], faiss.ScalarQuantizer.QT_8bit, faiss.METRIC_INNER_PRODUCT
            ),
            faiss.IndexPQ(matrix.shape[1], 8, 3, faiss.METRIC_INNER_PRODUCT),
        ]:
            index.train(matrix)
            index.add(matrix)
            _, labels = index.search(q[None, :], 10)
            print(type(index).__name__, "bytes per code", index.code_size, "first rows", labels[0])
            assert len(set(labels[0])) == 10

    if __name__ == "__main__":
        main()

## 05_sqlite_binary

Source: `modules/08_quantization/examples/05_sqlite_binary.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Compute a binary distance using sqlite-vec.

    Run: uv run --extra embedded python modules/08_quantization/examples/05_sqlite_binary.py
    Expected: compare the reported ranking and verify the final assertion.
    CPU: seconds, included data, no Docker or model downloads.
    """

    # course-extra: embedded
    import sqlite3

    import sqlite_vec

    from common.index_lessons import unit_catalog

    def main():
        _, matrix, q = unit_catalog(64)
        with sqlite3.connect(":memory:") as db:
            db.enable_load_extension(True)
            sqlite_vec.load(db)
            db.enable_load_extension(False)
            value = db.execute(
                "select vec_distance_hamming(vec_quantize_binary(?),vec_quantize_binary(?))",
                (matrix[0].tobytes(), q.tobytes()),
            ).fetchone()[0]
            print("different sign bits", value)
            assert 0 <= value <= matrix.shape[1]

    if __name__ == "__main__":
        main()

## 06_lance_prefix

Source: `modules/08_quantization/examples/06_lance_prefix.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Store a shortened Qwen prefix and retrieve its nearest LanceDB product.
    Run: uv run --extra embedded python modules/08_quantization/examples/06_lance_prefix.py
    CPU: a few seconds. Uses included Qwen samples, no model downloads or Docker.
    """

    # course-extra: embedded
    import tempfile

    import lancedb
    import numpy as np
    import pyarrow as pa

    from common.index_lessons import prefix_results
    from common.settings import ROOT

    def main():
        dimension = 64
        expected = prefix_results()[str(dimension)]["top10"]
        with np.load(ROOT / "data/tier2/matryoshka.npz", allow_pickle=False) as sample:
            vectors = sample["documents"][:, :dimension]
            vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
            query = sample["query"][:dimension]
            query = query / np.linalg.norm(query)
            schema = pa.schema([("id", pa.string()), ("vector", pa.list_(pa.float32(), dimension))])
            rows = [
                {"id": key, "vector": v.tolist()}
                for key, v in zip(sample["ids"].tolist(), vectors, strict=True)
            ]
        with tempfile.TemporaryDirectory() as directory:
            db = lancedb.connect(directory)
            table = db.create_table("qwen_prefix", data=pa.Table.from_pylist(rows, schema=schema))
            hits = table.search(query).distance_type("cosine").limit(10).to_list()
            assert {h["id"] for h in hits} == set(expected)
            print("Qwen 64-dimensional prefix", [(h["id"], h["_distance"]) for h in hits])
            del table, db

    if __name__ == "__main__":
        main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.index_lessons import prefix_results

print(prefix_results()["64"])

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.